In [1]:
import requests
import json

ANKI_URL = "http://localhost:8765"

def get_anki_vocab(limit_days=7):
    # 1. Notizen-IDs der letzten X Tage abfragen
    payload = {
        "action": "findNotes",
        "version": 6,
        "params": {"query": f"added:{limit_days}"}
    }
    res = requests.post(ANKI_URL, json=payload).json()
    note_ids = res.get("result", [])
    
    if not note_ids:
        print("Keine neuen Vokabeln gefunden.")
        return []

    # 2. Details zu diesen Notizen abrufen
    details_payload = {
        "action": "notesInfo",
        "version": 6,
        "params": {"notes": note_ids}
    }
    notes_info = requests.post(ANKI_URL, json=details_payload).json().get("result", [])
    
    vocab_list = []
    for note in notes_info:
        fields = note.get("fields", {})
        # ACHTUNG: Passe 'Front' und 'Back' an deine Anki-Feldnamen an (z. B. 'Türkisch', 'Deutsch')
        front = fields.get("Front", {}).get("value", "").strip()
        back = fields.get("Back", {}).get("value", "").strip()
        
        if front and back:
            vocab_list.append({"tr": front, "de": back})
            
    return vocab_list

# Testen
vokabeln = get_anki_vocab(limit_days=7)
print(f"Gefundene Vokabeln ({len(vokabeln)}):")
print(vokabeln[:5]) # Zeigt die ersten 5 Vokabeln


Gefundene Vokabeln (0):
[]


In [13]:
# Nextcloud Zugangsdaten
WEBDAV_URL = "https://tubcloud.tu-berlin.de/remote.php/dav/files/6c15e8e8-a12e-103c-98d9-7f3a285a3c9f" # Ohne Slash am Ende
USERNAME = "b.wolbring"
APP_PASSWORD = "pm3q8-Qfwxz-8eRQB-PFHxD-8HNcJ" # In Nextcloud unter Einstellungen -> Sicherheit -> App-Passwort

def upload_vocab_to_nextcloud(vocab_data):
    webdav_url = f"{WEBDAV_URL.strip('/')}/anki_vocab.json"
    
    response = requests.put(
        webdav_url,
        data=json.dumps(vocab_data, ensure_ascii=False).encode('utf-8'),
        auth=(USERNAME, APP_PASSWORD),
        headers={"Content-Type": "application/json"}
    )
    
    if response.status_code in [200, 201, 204]:
        print("✅ Vokabeln erfolgreich in die Nextcloud hochgeladen!")
    else:
        print(f"❌ Fehler beim Hochladen: {response.status_code} - {response.text}")

# Testen
upload_vocab_to_nextcloud(vokabeln)


✅ Vokabeln erfolgreich in die Nextcloud hochgeladen!


In [22]:
from google import genai
from google.genai import types

# Deinen API-Key von aistudio.google.com
GEMINI_API_KEY = "AQ.Ab8RN6IEUjYRk7-brzXFIjVKV3QHb8R5Q8x8kzsVqC3iHfbH9g"

client = genai.Client(api_key=GEMINI_API_KEY)

def generate_agent_response(user_input, vocab_list):
    # System Instruction zusammenbauen
    vocab_formatted = "\n".join([f"- {v['tr']} ({v['de']})" for v in vocab_list])
    
    system_instruction = f"""
    Du bist ein freundlicher und geduldiger Türkisch-Lehrer. Führe ein natürliches Alltagsgespräch auf Türkisch mit mir.
    
    WICHTIG: Baue bevorzugt folgende Vokabeln ein, die ich gerade lerne:
    {vocab_formatted}
    
    Verhaltensregeln:
    1. Antworte immer zuerst auf Türkisch.
    2. Verwende eine einfache, verständliche Sprache.
    3. Falls ich Grammatik- oder Vokabelfehler im Türkischen mache, korrigiere mich kurz auf Deutsch am Ende deiner Antwort.
    4. Beende deine Antwort immer mit einer offenen Frage auf Türkisch, damit das Gespräch weitergeht.
    """
    
    # Hier 'gemini-2.0-flash' nutzen (oder das Modell aus der Liste oben)
    response = client.models.generate_content(
        model="models/gemini-3.6-flash", 
        contents=user_input,
        config=types.GenerateContentConfig(
            system_instruction=system_instruction,
            temperature=0.7
        )
    )
    return response.text

# Testen
antwort = generate_agent_response("Hallo! Ich möchte heute Türkisch üben.", vokabeln)
print(antwort)

Merhaba! Hoş geldin! Çok sevindim. Bugün birlikte Türkçe pratik yapacağız. Harika bir karar! 

Nasılsın? Bugün günün nasıl geçiyor?


In [23]:
history = []

def chat_step(user_msg):
    print(f"Du: {user_msg}\n")
    agent_msg = generate_agent_response(user_msg, vokabeln)
    print(f"Gemini Agent:\n{agent_msg}")

# Beispielaufruf:
chat_step("Merhaba! Bugün nasılsın?")

Du: Merhaba! Bugün nasılsın?

Gemini Agent:
Merhaba! Ben çok iyiyim, teşekkür ederim. Bugün hava çok güzel ve seninle Türkçe konuştuğum için çok mutluyum. 

Sen bugün nasılsın? Bugün neler yapıyorsun, günün nasıl geçiyor?


In [18]:
for m in client.models.list():
    if "generateContent" in m.supported_actions:
        print(m.name)
        

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/lyria-3.5
models/gemini-3.1-flash-tts-preview
models/